In [1]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

In [2]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [3]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("season_database").getOrCreate()

In [4]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events_2022_2023 = spark.read.parquet(*season_events_parquet_file_paths)

df_events_2022_2023 = df_events_2022_2023.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("visibility", StringType(), True)
    ]))

df_events_2022_2023 = df_events_2022_2023.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": from_json("balls", balls_schema)

}).drop('homePlayers', 'awayPlayers', 'balls')

df_events_2022_2023.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------+----------------+------------+--------------+--------------------+--------------------+--------------------+
|             eventId|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|             details|eventPlayer.id|eventPlayer.name|eventTeam.id|eventTeam.name|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------+----------------+------------+--------------+--------------------+--------------------+--------------------+
|9e6a498f54bad910e...|            1|  4438|2022-2023|     1|       First half|       

In [5]:
print('Quantidade de linhas:', df_events_2022_2023.count())

Quantidade de linhas: 945154


In [6]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_2022_2023 = df_games.withColumnRenamed("id","gameId").filter(col('season') == '2022-2023').drop('season')

df_games_2022_2023.show(5)

+------+----------+----------------------+-------------+-------------+-------+--------------+--------------+----------------+---------------+--------------------+-------------+--------------+-------------+
|gameId|      date|teamExtraTimeStartSide|teamStartSide|    venueType|team.id|     team.name|competition.id|competition.name|opponentTeam.id|   opponentTeam.name| stadium.name|stadium.length|stadium.width|
+------+----------+----------------------+-------------+-------------+-------+--------------+--------------+----------------+---------------+--------------------+-------------+--------------+-------------+
|  4447|2022-08-13|                 Right|         Left|    TEAM_HOME|      3|   Aston Villa|             1|  Premier League|              8|             Everton|   Villa Park|         105.0|         68.0|
|  4760|2023-04-25|                  Left|         Left|OPPONENT_HOME|      7|Crystal Palace|             1|  Premier League|             20|Wolverhampton Wan...|     Molineux|

In [7]:
# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)

df_games_2022_2023 = (
    df_games_2022_2023.withColumns({
    "homeTeam.id": when(col('venueType') == 'TEAM_HOME', col('`team.id`')).otherwise(col('`opponentTeam.id`')),
    "homeTeam.name": when(col('venueType') == 'TEAM_HOME', col('`team.name`')).otherwise(col('`opponentTeam.name`')),

    "opponentTeam.id": when(col('venueType') == 'TEAM_HOME', col('`opponentTeam.id`')).otherwise(col('`opponentTeam.id`')),
    "opponentTeam.name": when(col('venueType') == 'TEAM_HOME', col('`opponentTeam.name`')).otherwise(col('`opponentTeam.name`')),

}).drop('venueType', 'team.id', 'team.name')
)

df_games_2022_2023.show(5)

+------+----------+----------------------+-------------+--------------+----------------+---------------+--------------------+-------------+--------------+-------------+-----------+--------------------+
|gameId|      date|teamExtraTimeStartSide|teamStartSide|competition.id|competition.name|opponentTeam.id|   opponentTeam.name| stadium.name|stadium.length|stadium.width|homeTeam.id|       homeTeam.name|
+------+----------+----------------------+-------------+--------------+----------------+---------------+--------------------+-------------+--------------+-------------+-----------+--------------------+
|  4447|2022-08-13|                 Right|         Left|             1|  Premier League|              8|             Everton|   Villa Park|         105.0|         68.0|          3|         Aston Villa|
|  4760|2023-04-25|                  Left|         Left|             1|  Premier League|             20|Wolverhampton Wan...|     Molineux|         105.0|         68.0|         20|Wolverhampto

In [8]:
df_events_games_2022_2023 = df_events_2022_2023.join(df_games_2022_2023, on = "gameId", how='left')
df_events_games_2022_2023.show()

+------+--------------------+-------------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------+----------------+------------+---------------+--------------------+--------------------+--------------------+----------+----------------------+-------------+--------------+----------------+---------------+-----------------+----------------+--------------+-------------+-----------+---------------+
|gameId|             eventId|competitionId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|             details|eventPlayer.id|eventPlayer.name|eventTeam.id| eventTeam.name|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|teamExtraTimeStartSide|teamStartSide|competition.id|competition.name|opponentTeam.id|opponentTeam.name|    stadium.name|stadium.length|stadium.width|homeTeam.id|  homeTeam.name|
+------+----

In [9]:
df_events_games_2022_2023 = df_events_games_2022_2023.withColumns({

    # Confere se nos dados de tracking do mandante não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_home":
    ((size(col("homePlayers_parsed")) > 0) & 
    forall(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_away":
    ((size(col("awayPlayers_parsed")) > 0) & 
    forall(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    "all_balls": 
    ((size(col("balls_parsed")) > 0) & 
    forall(
        col("balls_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Traz a quantidade de dicionários de cada evento para saber se tem 11 jogadores do time mandante e adversário
    "len_tracking_home": size(col("homePlayers_parsed")),
    "len_tracking_away": size(col("awayPlayers_parsed"))
})

df_events_games_2022_2023.show()

+------+--------------------+-------------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------+----------------+------------+---------------+--------------------+--------------------+--------------------+----------+----------------------+-------------+--------------+----------------+---------------+-----------------+----------------+--------------+-------------+-----------+---------------+-----------------+-----------------+---------+-----------------+-----------------+
|gameId|             eventId|competitionId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|             details|eventPlayer.id|eventPlayer.name|eventTeam.id| eventTeam.name|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|teamExtraTimeStartSide|teamStartSide|competition.id|competition.name|opponentTeam.id|opponentTeam.name|    s

In [10]:
df_acima_abaixo_11 = df_events_games_2022_2023.filter(
    ((col('len_tracking_home') != 11) & (col('len_tracking_home') > 0)) | 
    ((col('len_tracking_away') != 11) & (col('len_tracking_away') > 0)))

df_acima_abaixo_11.cache()
df_acima_abaixo_11.show()

+------+--------------------+-------------+---------+------+-----------------+-------------+--------------------+--------------+-----------------------+--------+--------------------+--------------+----------------+------------+--------------------+--------------------+--------------------+--------------------+----------+----------------------+-------------+--------------+----------------+---------------+--------------------+------------------+--------------+-------------+-----------+--------------------+-----------------+-----------------+---------+-----------------+-----------------+
|gameId|             eventId|competitionId|   season|period|periodDescription|    eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|             details|eventPlayer.id|eventPlayer.name|eventTeam.id|      eventTeam.name|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|teamExtraTimeStartSide|teamStartSide|competition.id|competition.name|opponentTeam.id| 

In [11]:
pandas_df = df_acima_abaixo_11.toPandas()
pandas_df.head()

,gameId,eventId,competitionId,season,period,periodDescription,eventType,eventTypeDescription,startGameClock,startFormattedGameClock,homeTeam,details,eventPlayer.id,eventPlayer.name,eventTeam.id,eventTeam.name,homePlayers_parsed,awayPlayers_parsed,balls_parsed,date,teamExtraTimeStartSide,teamStartSide,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width,homeTeam.id,homeTeam.name,all_tracking_home,all_tracking_away,all_balls,len_tracking_home,len_tracking_away
0,4466,bffc1e613bc0812dbaca18131d9ba966,1,2022-2023,2,Second half,OTB,A possession with a player on the ball,5296,88:16,False,"{""earlyDistribution"": false, ""endType"": null, ...",393,Harrison Reed,54,Fulham,"[(15.472999572753906, -22.391000747680664, (20...","[(-6.203000068664551, 14.90999984741211, (1973...","[(-15.819999694824219, 7.519999980926514, ESTI...",2022-08-27,Right,Left,1,Premier League,54,Fulham,Emirates Stadium,105.0,68.0,2,Arsenal,1,1,1,11,12
1,4466,a6f9132392697a99a00c7e6408960928,1,2022-2023,2,Second half,PA,Pass,5296,88:16,False,"{""blockerPlayer"": {""id"": null, ""name"": null, ""...",393,Harrison Reed,54,Fulham,"[(15.472999572753906, -22.391000747680664, (20...","[(-6.203000068664551, 14.90999984741211, (1973...","[(-15.819999694824219, 7.519999980926514, ESTI...",2022-08-27,Right,Left,1,Premier League,54,Fulham,Emirates Stadium,105.0,68.0,2,Arsenal,1,1,1,11,12
2,4522,9f53766958b4b86af504ece590f6eff0,1,2022-2023,2,Second half,SECONDKICKOFF,Second half kick off,2700,45:00,False,"{""earlyDistribution"": false, ""endType"": null, ...",448,Pascal Groß,4,Brighton & Hove Albion,"[(37.97200012207031, 0.6549999713897705, (32, ...","[(-39.404998779296875, 0.47699999809265137, (1...","[(-4.849999904632568, 1.6399999856948853, VISI...",2022-10-01,Left,Right,1,Premier League,10,Liverpool,Anfield,101.0,68.0,10,Liverpool,1,1,1,12,11
3,4522,76ec6d8a918207bce5cc6bd8a4ffd180,1,2022-2023,2,Second half,PA,Pass,2700,45:00,False,"{""blockerPlayer"": {""id"": null, ""name"": null, ""...",448,Pascal Groß,4,Brighton & Hove Albion,"[(37.97200012207031, 0.6549999713897705, (32, ...","[(-39.404998779296875, 0.47699999809265137, (1...","[(-4.849999904632568, 1.6399999856948853, VISI...",2022-10-01,Left,Right,1,Premier League,10,Liverpool,Anfield,101.0,68.0,10,Liverpool,1,1,1,12,11
4,4565,fad6cf30f2353bd0ba8383dfa0337e64,1,2022-2023,2,Second half,PA,Pass,4807,80:07,True,"{""blockerPlayer"": {""id"": null, ""name"": null, ""...",4915,Hee-chan Hwang,20,Wolverhampton Wanderers,"[(-13.730999946594238, 16.288999557495117, (23...","[(14.609999656677246, -3.2730000019073486, (41...",[],2022-10-23,Left,Left,1,Premier League,20,Wolverhampton Wanderers,Molineux,105.0,68.0,20,Wolverhampton Wanderers,1,1,0,13,11


In [12]:
x_homePlayers = []
y_homePlayers = []

x_awayPlayers = []
y_awayPlayers = []

x_ball = []
y_ball = []

for i, row in pandas_df.iterrows():
    for coords in row['homePlayers_parsed']:
        x_homePlayers.append(coords.x)
        y_homePlayers.append(coords.y)

    for coords in row['awayPlayers_parsed']:
        x_awayPlayers.append(coords.x)
        y_awayPlayers.append(coords.y)

    for coords in row['balls_parsed']:
        x_ball.append(coords.x)
        y_ball.append(coords.y)

    break

print('x_homePlayers', x_homePlayers)
print('y_homePlayers', y_homePlayers)
print('x_awayPlayers', x_awayPlayers)
print('y_awayPlayers', y_awayPlayers)
print('x_ball', x_ball)
print('y_ball', y_ball)

x_homePlayers [15.472999572753906, 39.29399871826172, -1.3020000457763672, 4.120999813079834, 18.788999557495117, 9.567999839782715, 16.180999755859375, 3.3910000324249268, 11.982000350952148, 4.478000164031982, 16.143999099731445]
y_homePlayers [-22.391000747680664, -3.2190001010894775, -1.4880000352859497, 4.395999908447266, -10.564000129699707, 2.046999931335449, 20.863000869750977, -26.086000442504883, -6.804999828338623, 15.737000465393066, 2.4709999561309814]
x_awayPlayers [-6.203000068664551, -13.10099983215332, -10.300999641418457, 4.209000110626221, 11.696000099182129, -8.857999801635742, -0.15800000727176666, 0.49300000071525574, 9.248000144958496, -34.64500045776367, -4.559000015258789, 0.25099998712539673]
y_awayPlayers [14.90999984741211, 7.85099983215332, 1.2380000352859497, -17.85700035095215, -25.575000762939453, -6.5320000648498535, -2.5399999618530273, 8.329000473022461, 29.323999404907227, -0.13300000131130219, -23.966999053955078, -5.195000171661377]
x_ball [-15.819

In [13]:
pandas_df['awayPlayers_parsed'].iloc[0]

[Row(x=-6.203000068664551, y=14.90999984741211, player=Row(id=1973, name='Antonee Robinson'), visibility='ESTIMATED', confidence='LOW', jerseyNum=None),
 Row(x=-13.10099983215332, y=7.85099983215332, player=Row(id=257, name='Issa Diop'), visibility='ESTIMATED', confidence='LOW', jerseyNum=None),
 Row(x=-10.300999641418457, y=1.2380000352859497, player=Row(id=1971, name='Tim Ream'), visibility='ESTIMATED', confidence='LOW', jerseyNum=None),
 Row(x=4.209000110626221, y=-17.85700035095215, player=Row(id=1990, name='Bobby Reid'), visibility='ESTIMATED', confidence='LOW', jerseyNum=None),
 Row(x=11.696000099182129, y=-25.575000762939453, player=Row(id=1991, name='Aleksandar Mitrovic'), visibility='ESTIMATED', confidence='LOW', jerseyNum=None),
 Row(x=-8.857999801635742, y=-6.5320000648498535, player=Row(id=13, name='Tosin Adarabioyo'), visibility='ESTIMATED', confidence='LOW', jerseyNum=None),
 Row(x=-0.15800000727176666, y=-2.5399999618530273, player=Row(id=393, name='Harrison Reed'), visi

In [ ]:
# montar base com tracking dos 22 e bola -> analisar primeiro casos com menos ou mais de 11 dados de tracking
#registros com mais ou menos que 11 x,y do mandante e adversário -> corrigir pra concertar a tabela
#jogadores muito dispersos
#jogadores muito perto um do outro
#% de valores estimados
#% da confiança dos valores